# Phase 3 — Deep EDA: what separates fraud from legit?

We work on the **cleaned (deduped)** data and on **raw, unscaled** values so numbers stay human-readable (real hours, real dollars).

Goal: 2–3 *honest, quotable* insights — not a wall of charts.

⚠️ **Limitation up front:** V1–V28 are PCA-anonymized. We can rank which features separate the classes, but we cannot say what they *mean* in business terms. We will not invent a story for them.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_data
from src.preprocessing import clean_data
from src import eda

df = clean_data(load_data())   # deduped, raw-scale
df.shape

Loaded 284,807 rows x 31 columns
Fraud rate: 0.1727%  (492 fraud cases)


Dropped 1,081 duplicate rows (283,726 remain)


(283726, 31)

## 1. Does fraud cluster in time?
Fraud **rate** by hour (not raw count — count just tracks overall traffic).

In [2]:
by_hour = eda.fraud_rate_by_hour(df)
by_hour[['n', 'fraud', 'fraud_rate']].sort_values('fraud_rate', ascending=False).head()
# Figure saved to reports/figures/03_fraud_rate_by_hour.png

,n,fraud,fraud_rate
Hour,,,
2.0,3308,48,0.014510
4.0,2204,23,0.010436
3.0,3487,17,0.004875
5.0,2988,11,0.003681
7.0,7233,23,0.003180


**Finding:** fraud rate spikes in the small hours — hour 2 (~1.45%) and hour 4 (~1.04%) run ~6–9x the overall average of 0.167%. Volume is *lowest* then, but each transaction is far riskier. A real fraud team could weight night-time transactions more heavily for review.

## 2. Which features separate the classes?

In [3]:
eda.feature_separation(df, top=8).round(3)

V17   -0.313
V14   -0.293
V12   -0.251
V10   -0.207
V16   -0.187
V3    -0.182
V7    -0.172
V11    0.149
Name: Class, dtype: float64

In [4]:
eda.boxplots_top_features(df)     # -> 04_top_feature_boxplots.png
eda.correlation_heatmap(df)       # -> 05_corr_with_target.png
print('Figures saved to reports/figures/')

Figures saved to reports/figures/


**Finding:** V17, V14, V12, V10 are the strongest separators (|corr| 0.21–0.31), all *negatively* correlated — fraud cases push sharply negative on these components, while legit transactions cluster tightly near zero. The boxplots show almost no overlap in the bulk of the distributions, which is promising for a classifier. (We still can't name what V17 represents — PCA.)

Note: `Amount` and `Time` have near-zero linear correlation with fraud (~0.01). Linear correlation misses nonlinear signal — tree models in Phase 6 may still use them.

## 📝 Phase 3 takeaways
1. **Time:** fraud rate peaks 2–4am (~6–9x baseline) despite low volume → night transactions are riskier per-unit.
2. **Features:** V17/V14/V12/V10 separate the classes most; fraud skews strongly negative on them.
3. **Honesty:** PCA columns can't be interpreted in business terms; `Amount`/`Time` show no *linear* fraud signal.

**Next — Phase 4:** feature engineering (Hour, log-Amount, high-amount flag) before we touch the imbalance problem.